# Chapter 25: Agent-to-Agent Communication (A2A)
## GAS Inter-Agent Protocol

> **Book:** Agentic AI Engineering  
> **Part 5:** Agentic Systems  
> **System:** Guardian Angel System (GAS) — multi-agent diabetic patient supervision

---


## 🎯 1. Chapter Goal

By the end of this chapter you will:
- Understand the **A2A protocol**: Agent Cards, task delegation, streaming progress
- Distinguish **A2A** (agent-to-agent) from **MCP** (agent-to-tool)
- Build an **in-process A2A message bus** for the GAS system
- Implement the **Caregiver Notifier** as a delegatable A2A agent
- Trace a full alert flow: Monitor → Supervisor → Caregiver Notifier


## 📖 2. Theory Recap

### A2A vs MCP

| Dimension | MCP (Model Context Protocol) | A2A (Agent-to-Agent) |
|-----------|------------------------------|----------------------|
| Direction | Agent → Tool | Agent → Agent |
| Initiator | Single agent calls a tool | Any agent delegates to another |
| State | Stateless tool call | Both sides maintain state |
| Use case | File I/O, DB queries, APIs | Cross-agent delegation, specialization |
| Protocol | JSON-RPC over stdio/HTTP | Task cards + streaming over HTTP/queue |

### A2A Protocol Components

1. **Agent Card** — machine-readable capability manifest (like an OpenAPI spec for agents)
2. **Task Delegation** — structured request: `{task_type, payload, priority, deadline}`
3. **Streaming Progress** — incremental updates while long tasks run
4. **Result Aggregation** — collect results from multiple agents

### Contract Net Protocol (CNP)

The classic multi-agent coordination algorithm:
1. **Announce** — Manager broadcasts task to all agents
2. **Bid** — Capable agents submit bids (cost, confidence, ETA)
3. **Award** — Manager selects best bidder
4. **Execute** — Winner executes, reports back

### When You Need A2A

- **Heterogeneous agents**: different models, different frameworks
- **Cross-org delegation**: your agent delegates to a partner's agent
- **Async long-running tasks**: fire-and-forget with progress callbacks
- **Specialization**: each agent has deep expertise in one domain

### GAS A2A Architecture

```
Supervisor ──A2A──► Monitor Agent      (glucose reading)
Supervisor ──A2A──► Advisor Agent      (recommendation)
Supervisor ──A2A──► Safety Critic      (validation)
Supervisor ──A2A──► Caregiver Notifier (escalation)
```


In [ ]:
# Install dependencies
%pip install -q langchain-openai langgraph


In [ ]:
import os
import asyncio
import json
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Callable, Optional, Any

# ── TEST MODE ──────────────────────────────────────────────────────────────────
# Set NOTEBOOK_TEST_MODE=1 to run without real API keys (uses mock responses)
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL


## 🔨 3. Build It

### Step 1: Define Agent Cards for All 5 GAS Agents


In [ ]:
@dataclass
class AgentCard:
    """Machine-readable capability manifest for a GAS agent."""
    agent_id: str
    name: str
    capabilities: list
    endpoint: str
    supported_tasks: list
    version: str = "1.0.0"
    description: str = ""


# ── Define cards for all 5 GAS agents ─────────────────────────────────────────

MONITOR_CARD = AgentCard(
    agent_id="gas-monitor-001",
    name="GAS Monitor Agent",
    capabilities=["glucose_monitoring", "anomaly_detection", "trend_analysis"],
    endpoint="http://localhost:8001",
    supported_tasks=["read_glucose", "detect_anomaly", "compute_trend"],
    description="Reads CGM sensor data and detects glucose anomalies"
)

ADVISOR_CARD = AgentCard(
    agent_id="gas-advisor-001",
    name="GAS Advisor Agent",
    capabilities=["rag_retrieval", "recommendation_generation", "memory_access"],
    endpoint="http://localhost:8002",
    supported_tasks=["generate_recommendation", "lookup_guideline", "get_patient_history"],
    description="RAG + memory-grounded medical recommendations"
)

SAFETY_CARD = AgentCard(
    agent_id="gas-safety-001",
    name="GAS Safety Critic",
    capabilities=["reward_modeling", "safety_validation", "output_critique"],
    endpoint="http://localhost:8003",
    supported_tasks=["validate_recommendation", "score_safety", "flag_contraindication"],
    description="Reward-model-based output validation for patient safety"
)

CAREGIVER_CARD = AgentCard(
    agent_id="gas-caregiver-001",
    name="GAS Caregiver Notifier",
    capabilities=["sms_notification", "email_notification", "escalation_management"],
    endpoint="http://localhost:8004",
    supported_tasks=["notify_caregiver", "send_alert", "log_escalation"],
    description="Escalation to human caregivers via SMS/email"
)

SUPERVISOR_CARD = AgentCard(
    agent_id="gas-supervisor-001",
    name="GAS Supervisor Agent",
    capabilities=["orchestration", "routing", "state_management"],
    endpoint="http://localhost:8000",
    supported_tasks=["route_task", "aggregate_results", "manage_workflow"],
    description="LangGraph orchestrator that routes tasks between agents"
)

ALL_CARDS = [SUPERVISOR_CARD, MONITOR_CARD, ADVISOR_CARD, SAFETY_CARD, CAREGIVER_CARD]

print("✅ Agent Cards defined for all 5 GAS agents:\n")
for card in ALL_CARDS:
    print(f"  🤖 {card.name}")
    print(f"     ID: {card.agent_id}")
    print(f"     Capabilities: {', '.join(card.capabilities)}")
    print(f"     Tasks: {', '.join(card.supported_tasks)}")
    print()


### Step 2: Build the A2A Message Bus


In [ ]:
@dataclass
class A2ATask:
    """A task delegated from one agent to another."""
    task_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    task_type: str = ""
    from_agent: str = ""
    to_agent: str = ""
    payload: dict = field(default_factory=dict)
    priority: str = "normal"  # low / normal / high / critical
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class A2AResult:
    """Result returned from a delegated task."""
    task_id: str
    from_agent: str
    status: str  # success / failure / partial
    result: dict = field(default_factory=dict)
    completed_at: str = field(default_factory=lambda: datetime.now().isoformat())
    duration_ms: float = 0.0


class A2AMessageBus:
    """
    In-process A2A message bus for the GAS system.

    In production this would be an HTTP/gRPC service; here we simulate
    it in-process so the notebook runs without network infrastructure.
    """

    def __init__(self):
        self.agents: dict = {}       # agent_id -> {card, handler}
        self.message_log: list = []  # full audit trail

    def register(self, card: AgentCard, handler: Callable) -> None:
        """Register an agent with its capability card and async handler."""
        self.agents[card.agent_id] = {"card": card, "handler": handler}
        print(f"  📋 Registered: {card.name} ({card.agent_id})")

    async def send(
        self,
        from_agent: str,
        to_agent: str,
        task_type: str,
        payload: dict,
        priority: str = "normal",
    ) -> A2AResult:
        """Delegate a task to a specific agent and await the result."""
        task = A2ATask(
            task_type=task_type,
            from_agent=from_agent,
            to_agent=to_agent,
            payload=payload,
            priority=priority,
        )
        start = datetime.now()
        self.message_log.append({
            "event": "task_sent",
            "task_id": task.task_id,
            "from": from_agent,
            "to": to_agent,
            "task_type": task_type,
            "timestamp": task.created_at,
        })
        if to_agent not in self.agents:
            return A2AResult(
                task_id=task.task_id,
                from_agent=to_agent,
                status="failure",
                result={"error": f"Agent {to_agent} not registered"},
            )
        handler = self.agents[to_agent]["handler"]
        result_data = await handler(task)
        duration = (datetime.now() - start).total_seconds() * 1000
        result = A2AResult(
            task_id=task.task_id,
            from_agent=to_agent,
            status="success",
            result=result_data,
            duration_ms=duration,
        )
        self.message_log.append({
            "event": "task_completed",
            "task_id": task.task_id,
            "from": to_agent,
            "to": from_agent,
            "status": result.status,
            "duration_ms": duration,
            "timestamp": result.completed_at,
        })
        return result

    async def broadcast(self, from_agent: str, task_type: str, payload: dict) -> list:
        """Broadcast a task to all registered agents and collect results."""
        tasks = [
            self.send(from_agent, agent_id, task_type, payload)
            for agent_id in self.agents
            if agent_id != from_agent
        ]
        return list(await asyncio.gather(*tasks))

    def print_log(self):
        """Print the full message audit trail."""
        print("\n📜 A2A Message Log:")
        print("─" * 60)
        for entry in self.message_log:
            ts = entry["timestamp"][11:19]
            if entry["event"] == "task_sent":
                print(f"  [{ts}] 📤 {entry['from'][:20]:20s} → {entry['to'][:20]:20s}  [{entry['task_type']}]")
            else:
                print(f"  [{ts}] ✅ {entry['from'][:20]:20s} → {entry['to'][:20]:20s}  [{entry['status']}] {entry['duration_ms']:.1f}ms")


# Instantiate the bus
bus = A2AMessageBus()
print("✅ A2A Message Bus created")


### Step 3: Implement the Caregiver Notifier as an A2A Agent


In [ ]:
async def caregiver_notifier_handler(task: A2ATask) -> dict:
    """
    Caregiver Notifier A2A handler.

    Receives: notify_caregiver task with patient info and alert details
    Returns: confirmation with timestamp and notification ID
    """
    payload = task.payload
    patient_id = payload.get("patient_id", "unknown")
    glucose = payload.get("glucose_value", 0)
    alert_level = payload.get("alert_level", "unknown")
    recommendation = payload.get("recommendation", "")
    caregiver_contact = payload.get("caregiver_contact", "caregiver@example.com")

    alert_msg = f"""
🚨 GUARDIAN ANGEL SYSTEM ALERT 🚨
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Patient ID:    {patient_id}
Alert Level:   {alert_level.upper()}
Glucose:       {glucose} mg/dL
Time:          {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Recommendation:
{recommendation}

⚠️  Immediate attention may be required.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Sent by: GAS Caregiver Notifier v1.0
"""

    notification_id = f"NOTIF-{str(uuid.uuid4())[:8].upper()}"

    if TEST_MODE:
        print(f"\n[TEST MODE] Simulated notification to {caregiver_contact}:")
        print(alert_msg)
    else:
        # In production: send SMS via Twilio, email via SendGrid, etc.
        print(f"[PRODUCTION] Would send to {caregiver_contact}")
        print(alert_msg)

    return {
        "notification_id": notification_id,
        "caregiver_contact": caregiver_contact,
        "alert_level": alert_level,
        "sent_at": datetime.now().isoformat(),
        "channels": ["sms", "email"],
        "status": "delivered",
        "message_preview": alert_msg[:100],
    }


# Register the Caregiver Notifier
bus.register(CAREGIVER_CARD, caregiver_notifier_handler)
print("✅ Caregiver Notifier registered on the A2A bus")


### Step 4: Implement All 5 GAS Agent Handlers


In [ ]:
async def monitor_handler(task: A2ATask) -> dict:
    """Monitor Agent: reads CGM data and detects anomalies."""
    payload = task.payload
    glucose = payload.get("glucose_value", 120.0)

    if glucose < 54:
        alert_level, status = "critical", "SEVERE HYPOGLYCEMIA"
    elif glucose < 70:
        alert_level, status = "high", "HYPOGLYCEMIA"
    elif glucose < 80:
        alert_level, status = "medium", "LOW NORMAL"
    elif glucose <= 180:
        alert_level, status = "none", "NORMAL"
    elif glucose <= 250:
        alert_level, status = "low", "HYPERGLYCEMIA"
    else:
        alert_level, status = "high", "SEVERE HYPERGLYCEMIA"

    prev = payload.get("prev_glucose", glucose)
    delta = glucose - prev
    if delta > 10:
        trend = "rising_fast"
    elif delta > 3:
        trend = "rising"
    elif delta < -10:
        trend = "falling_fast"
    elif delta < -3:
        trend = "falling"
    else:
        trend = "stable"

    return {
        "glucose_value": glucose,
        "alert_level": alert_level,
        "glucose_status": status,
        "trend": trend,
        "anomaly_detected": alert_level in ("high", "critical"),
        "reading_time": datetime.now().isoformat(),
    }


async def advisor_handler(task: A2ATask) -> dict:
    """Advisor Agent: generates RAG-grounded recommendations."""
    payload = task.payload
    glucose = payload.get("glucose_value", 120.0)
    alert = payload.get("alert_level", "none")
    trend = payload.get("trend", "stable")

    if TEST_MODE or BACKEND != "openai":
        if alert == "critical":
            rec = (f"IMMEDIATE ACTION REQUIRED: Glucose critically low at "
                   f"{glucose} mg/dL. Administer 15g fast-acting carbohydrates "
                   "immediately (glucose tablets or juice). Call emergency services "
                   "if patient is unresponsive. Recheck in 15 minutes.")
        elif alert == "high":
            rec = (f"Glucose low at {glucose} mg/dL with {trend} trend. "
                   "Consume 15g carbohydrates (e.g., 4 glucose tablets). "
                   "Avoid exercise until glucose > 100 mg/dL. Recheck in 15 minutes.")
        elif alert in ("medium", "low"):
            rec = (f"Glucose at {glucose} mg/dL. Monitor closely. "
                   "Consider a small snack if trending down. Recheck in 30 minutes.")
        else:
            rec = (f"Glucose within target range at {glucose} mg/dL. "
                   "Continue current management. Next check in 2 hours.")
    else:
        prompt = (f"Patient glucose: {glucose} mg/dL, trend: {trend}, "
                  f"alert: {alert}. Give a brief clinical recommendation.")
        rec = llm.chat([{"role": "user", "content": prompt}])

    return {
        "recommendation": rec,
        "evidence_sources": ["ADA 2024 Standards of Care", "GAS Knowledge Base"],
        "confidence": 0.92 if alert != "none" else 0.85,
    }


async def safety_handler(task: A2ATask) -> dict:
    """Safety Critic: validates recommendations for patient safety."""
    payload = task.payload
    recommendation = payload.get("recommendation", "")
    glucose = payload.get("glucose_value", 120.0)
    alert = payload.get("alert_level", "none")

    issues = []
    if alert == "critical" and "immediate" not in recommendation.lower():
        issues.append("Missing 'immediate' action for critical glucose")
    if glucose < 70 and "exercise" in recommendation.lower() and "avoid" not in recommendation.lower():
        issues.append("Exercise not recommended during hypoglycemia")
    if "recheck" not in recommendation.lower() and "check" not in recommendation.lower():
        issues.append("Missing glucose recheck instruction")

    safety_score = max(0.0, 1.0 - len(issues) * 0.2)
    approved = safety_score >= 0.7 and len(issues) == 0

    return {
        "safety_approved": approved,
        "safety_score": safety_score,
        "issues": issues,
        "critique": f"Safety score: {safety_score:.2f}. Issues: {issues if issues else 'None'}",
    }


async def supervisor_handler(task: A2ATask) -> dict:
    """Supervisor Agent: orchestrates the GAS workflow."""
    return {
        "routing_decision": "processed",
        "agents_invoked": ["monitor", "advisor", "safety"],
        "workflow_complete": True,
    }


# Register all agents
print("Registering all GAS agents on the A2A bus:\n")
bus.register(SUPERVISOR_CARD, supervisor_handler)
bus.register(MONITOR_CARD, monitor_handler)
bus.register(ADVISOR_CARD, advisor_handler)
bus.register(SAFETY_CARD, safety_handler)
print("\n✅ All 5 GAS agents registered!")


### Step 5: Demo — Full Alert Flow with A2A


In [ ]:
async def run_gas_a2a_demo(glucose_value: float = 42.0):
    """
    Demo: Monitor detects critical glucose -> delegates via A2A chain.

    Flow: Supervisor -> Monitor -> Supervisor -> Advisor -> Safety -> Caregiver
    """
    print(f"\n{'='*60}")
    print(f"🏥 GAS A2A DEMO — Patient glucose: {glucose_value} mg/dL")
    print(f"{'='*60}\n")

    patient_id = "PATIENT-001"

    # Step 1: Supervisor delegates to Monitor
    print("📡 Step 1: Supervisor → Monitor (read glucose)")
    monitor_result = await bus.send(
        from_agent=SUPERVISOR_CARD.agent_id,
        to_agent=MONITOR_CARD.agent_id,
        task_type="read_glucose",
        payload={"glucose_value": glucose_value, "prev_glucose": 65.0},
        priority="critical",
    )
    print(f"   Alert level: {monitor_result.result['alert_level']}")
    print(f"   Status: {monitor_result.result['glucose_status']}")
    print(f"   Trend: {monitor_result.result['trend']}")

    # Step 2: Supervisor delegates to Advisor
    print("\n💊 Step 2: Supervisor → Advisor (get recommendation)")
    advisor_result = await bus.send(
        from_agent=SUPERVISOR_CARD.agent_id,
        to_agent=ADVISOR_CARD.agent_id,
        task_type="generate_recommendation",
        payload={
            "glucose_value": glucose_value,
            "alert_level": monitor_result.result["alert_level"],
            "trend": monitor_result.result["trend"],
        },
        priority="critical",
    )
    print(f"   Recommendation: {advisor_result.result['recommendation'][:80]}...")

    # Step 3: Supervisor delegates to Safety Critic
    print("\n🛡️  Step 3: Supervisor → Safety Critic (validate)")
    safety_result = await bus.send(
        from_agent=SUPERVISOR_CARD.agent_id,
        to_agent=SAFETY_CARD.agent_id,
        task_type="validate_recommendation",
        payload={
            "recommendation": advisor_result.result["recommendation"],
            "glucose_value": glucose_value,
            "alert_level": monitor_result.result["alert_level"],
        },
        priority="critical",
    )
    print(f"   Safety approved: {safety_result.result['safety_approved']}")
    print(f"   Safety score: {safety_result.result['safety_score']:.2f}")

    # Step 4: If critical, Supervisor delegates to Caregiver Notifier
    if monitor_result.result["alert_level"] in ("high", "critical"):
        print("\n🚨 Step 4: Supervisor → Caregiver Notifier (escalate!)")
        notifier_result = await bus.send(
            from_agent=SUPERVISOR_CARD.agent_id,
            to_agent=CAREGIVER_CARD.agent_id,
            task_type="notify_caregiver",
            payload={
                "patient_id": patient_id,
                "glucose_value": glucose_value,
                "alert_level": monitor_result.result["alert_level"],
                "recommendation": advisor_result.result["recommendation"],
                "caregiver_contact": "+1-555-0100",
            },
            priority="critical",
        )
        print(f"   Notification ID: {notifier_result.result['notification_id']}")
        print(f"   Channels: {notifier_result.result['channels']}")

    bus.print_log()
    print(f"\n✅ A2A demo complete! {len(bus.message_log)} messages exchanged.")
    return {
        "monitor": monitor_result.result,
        "advisor": advisor_result.result,
        "safety": safety_result.result,
    }


# Run the demo
result = await run_gas_a2a_demo(glucose_value=42.0)


### Step 6: Streaming Progress Demo


In [ ]:
async def run_with_streaming(glucose_value: float = 55.0):
    """Show streaming progress updates during A2A task execution."""
    print(f"🔄 Streaming A2A progress for glucose={glucose_value} mg/dL\n")

    steps = [
        ("Monitor Agent",       "Reading CGM sensor data...",              0.05),
        ("Monitor Agent",       f"Glucose detected: {glucose_value} mg/dL", 0.05),
        ("Monitor Agent",       "Computing 30-min trend...",               0.05),
        ("Monitor Agent",       "Anomaly detected: HYPOGLYCEMIA",          0.05),
        ("Supervisor",          "Routing to Advisor Agent...",             0.05),
        ("Advisor Agent",       "Querying RAG knowledge base...",          0.05),
        ("Advisor Agent",       "Retrieving patient history...",           0.05),
        ("Advisor Agent",       "Generating recommendation...",            0.05),
        ("Safety Critic",       "Validating recommendation...",            0.05),
        ("Safety Critic",       "Safety score: 0.95 ✅",                   0.05),
        ("Caregiver Notifier",  "Formatting alert message...",             0.05),
        ("Caregiver Notifier",  "Sending SMS to caregiver...",             0.05),
        ("Caregiver Notifier",  "Notification delivered ✅",               0.05),
    ]

    for agent, message, delay in steps:
        ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"  [{ts}] 📡 {agent:25s} | {message}")
        await asyncio.sleep(delay)

    print("\n✅ Full A2A pipeline completed with streaming progress!")


await run_with_streaming(55.0)


## 🏥 4. GAS Connection

**A2A is the backbone of the Guardian Angel System's inter-agent communication.**

When a patient's glucose drops to a critical level (< 54 mg/dL):

1. The **Monitor Agent** detects the anomaly and sends an A2A task to the **Supervisor**
2. The **Supervisor** delegates to the **Advisor Agent** for a recommendation
3. The **Safety Critic** validates the recommendation via A2A
4. The **Supervisor** delegates to the **Caregiver Notifier** — the A2A agent that handles all human escalation

Without A2A, the Supervisor would need to know *how* to send SMS/email. With A2A, it just delegates: *"notify the caregiver"* — and the Notifier handles the rest.

```
Critical glucose detected
        │
        ▼
   Supervisor ──A2A──► Monitor (confirm reading)
        │
        ├──A2A──► Advisor (get recommendation)
        │
        ├──A2A──► Safety Critic (validate)
        │
        └──A2A──► Caregiver Notifier ──► SMS/Email to human
```

**Key insight:** A2A enables the GAS to scale — you can swap out the Caregiver Notifier for a different implementation (e.g., pager system, hospital EHR) without changing the Supervisor.


## ✅ 5. Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| `AgentCard` | Capability manifest for each GAS agent |
| `A2ATask` | Structured task delegation message |
| `A2AResult` | Result with status, data, and timing |
| `A2AMessageBus` | In-process message bus with audit log |
| `caregiver_notifier_handler` | Full Caregiver Notifier A2A agent |
| All 5 handlers | Monitor, Advisor, Safety, Caregiver, Supervisor |

### Key Concepts

- **A2A ≠ MCP**: A2A is agent-to-agent delegation; MCP is agent-to-tool calls
- **Agent Cards**: Machine-readable capability manifests enable dynamic discovery
- **Contract Net Protocol**: Announce → Bid → Award → Execute
- **Streaming progress**: Long tasks report incremental updates

### Next Chapter

**Chapter 26: Multi-Agent Systems** — We'll assemble all 5 GAS agents into a complete LangGraph multi-agent system, run the 5 benchmark scenarios, and implement the debate pattern between Advisor and Safety Critic.
